In [49]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/licenses_fall2022.csv"
df = pd.read_csv(url)

print(df.columns.tolist())

df.columns = df.columns.str.strip()

license_col = "Description"
county_col = "County"
status_col = "License Status"

df_clean = df.dropna(subset=[license_col, county_col, status_col]).copy()

df_clean[license_col] = df_clean[license_col].astype(str).str.strip()
df_clean[county_col] = df_clean[county_col].astype(str).str.strip()
df_clean[status_col] = df_clean[status_col].astype(str).str.strip()

df_clean = df_clean[
    (df_clean[license_col] != "") &
    (df_clean[county_col] != "") &
    (df_clean[status_col] != "")
]

top_types = (
    df_clean[license_col]
    .value_counts()
    .head(10)
    .reset_index()
)

top_types.columns = [license_col, "Count"]

vis1 = alt.Chart(top_types).mark_bar().encode(
    x = alt.X("Count:Q", title = "Number of Records"),
    y = alt.Y(f"{license_col}:N", sort = "-x", title = "License Description"),
    tooltip = [
        alt.Tooltip(f"{license_col}:N", title = "License Description"),
        alt.Tooltip("Count:Q", title = "Count")
    ]
).properties(
    title = "Top 10 License Descriptions",
    width = 700,
    height = 400
)

county_status = (
    df_clean.groupby([county_col, status_col])
    .size()
    .reset_index(name = "Count")
)

top_counties = (
    county_status.groupby(county_col)["Count"]
    .sum()
    .reset_index()
    .sort_values("Count", ascending = False)
    .head(15)[county_col]
)

county_status_top = county_status[county_status[county_col].isin(top_counties)]

status_options = sorted(county_status_top[status_col].dropna().unique().tolist())

status_dropdown = alt.binding_select(options = status_options, name = "License Status: ")

status_select = alt.param(
    name = "status_filter",
    bind = status_dropdown,
    value = status_options[0]
)

vis2 = alt.Chart(county_status_top).mark_bar().encode(
    x = alt.X("Count:Q", title = "Number of Records"),
    y = alt.Y(f"{county_col}:N", sort = "-x", title = "County"),
    color = alt.Color(f"{status_col}:N", title = "License Status"),
    tooltip=[
        alt.Tooltip(f"{county_col}:N", title = "County"),
        alt.Tooltip(f"{status_col}:N", title = "License Status"),
        alt.Tooltip("Count:Q", title = "Count")
    ]
).transform_filter(
    alt.datum[status_col] == status_select
).add_params(
    status_select
).properties(
    title="License Counts by County, Filtered by License Status",
    width = 700,
    height = 400
)

vis1.save("vis1.json")
vis2.save("vis2.json")

['_id', 'License Type', 'Description', 'License Number', 'License Status', 'Business', 'Title', 'First Name', 'Middle', 'Last Name', 'Prefix', 'Suffix', 'Business Name', 'BusinessDBA', 'Original Issue Date', 'Effective Date', 'Expiration Date', 'City', 'State', 'Zip', 'County', 'Specialty/Qualifier', 'Controlled Substance Schedule', 'Delegated Controlled Substance Schedule', 'Ever Disciplined', 'LastModifiedDate', 'Case Number', 'Action', 'Discipline Start Date', 'Discipline End Date', 'Discipline Reason']


In [47]:
vis1

alt.Chart(...)

In [48]:
vis2

alt.Chart(...)